# 02. 머신러닝 모델 학습 (Machine Learning Model Training)
월세 가격은 비단 '통학 거리'에 의해서만 결정되지 않습니다. 방의 '면적(크기)'과 '연식(건축년도)'이라는 강력한 가격 결정 변수들이 혼재되어 있기 때문입니다.

이에 본 프로젝트는 **HistGradientBoosting 회귀 알고리즘**을 도입하여, 여러 외부 요인들을 수학적으로 고정시킨 채 오직 **거리가 월세에 미치는 순수한 영향력(Partial Dependence)**을 추출해 냅니다.

###  단조 제약 조건 (Monotonic Constraints) 적용
모델이 노이즈나 특이치에 흔들리지 않고 부동산 시장의 보편적 상식을 준수하도록 다음과 같은 제약을 강제합니다.
- **임대면적 (+1)**: 평수가 넓을수록 가격은 반드시 상승해야 합니다.
- **건축년도 (+1)**: 최근에 지어진 신축일수록 가격은 반드시 상승해야 합니다.
- **통학거리 (0, 제약 없음)**: 본 프로젝트의 핵심입니다. 거리에 따른 가격 변동 패턴은 모델이 스스로 데이터에 내재된 '상권 프리미엄(비선형적 특성)'을 찾아내도록 어떠한 제약도 걸지 않습니다.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import os

# 전처리된 최종 데이터 로드
df = pd.read_csv("data/final_rent_data.csv")
df = df.dropna(subset=['임대면적', '직선거리(m)', '건축년도', '월실질주거비(만원)'])

# 학습에 사용할 피처 (면적, 거리, 년도)
features = ["임대면적", "직선거리(m)", "건축년도"]
target = "월실질주거비(만원)"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 단조 제약 조건 부여: 임대면적(+1), 직선거리(0), 건축년도(+1)
# 발표용 렌더링에 적합한 거시적 트렌드 도출을 위해 트리 깊이를 얕게 제한합니다.
model = HistGradientBoostingRegressor(
    max_iter=100,             # 반복 횟수 축소
    max_depth=4,              # 트리의 깊이를 얕게 제한 (계단 현상 완화)
    min_samples_leaf=30,      # 한 노드에 최소 30개 이상의 데이터가 있어야 분기 (특정 건물의 가격에 휘둘리는 것 방지)
    l2_regularization=1.0,    # L2 규제 적용
    random_state=42,
    monotonic_cst=[1, 0, 1] 
)

print("모델 학습 진행 중...")
model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f" 학습 완료! (평균 오차: {rmse:.2f} 만원)\n")

# 03번 분석 노트북에서 사용하기 위해 학습된 모델을 저장합니다.
os.makedirs("data", exist_ok=True)
joblib.dump(model, "data/rent_model.pkl")
print("모델이 'data/rent_model.pkl'로 저장되었습니다.")


FileNotFoundError: [Errno 2] No such file or directory: 'data/final_rent_data.csv'